# Session 3. The LangChain 1.x layer: create_agent, middleware, tool design

**Session 2's graph becomes one call. The graph is still underneath.**

- `create_agent` rebuilds the same agent; middleware hooks extend it
- tools designed by engineering criteria, and the JSON schema they ride in
- what a run costs, read as token counters

In [ ]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()  # reads .env once; nothing below opens a file


def chat_model(size: str = "cheap", **kwargs):
    """A model object for the configured provider. A dozen lines, copy them once."""
    name = os.environ[f"MODEL_{size.upper()}"]  # ids live in .env, never in code
    secret = os.environ["LLM_API_KEY"]
    if os.getenv("LLM_REASONING_EFFORT"):  # gpt-5.x: tools need reasoning "none"
        kwargs.setdefault("reasoning_effort", os.environ["LLM_REASONING_EFFORT"])
    # Gemini over OpenAI-compat drops the reasoning signature: turn two 400s
    if os.getenv("LLM_PROVIDER", "openai_compat") == "google_genai":
        return init_chat_model(f"google_genai:{name}", api_key=secret, **kwargs)
    return init_chat_model(
        f"openai:{name}", api_key=secret, base_url=os.environ["LLM_BASE_URL"], **kwargs
    )


print("provider:", os.getenv("LLM_PROVIDER", "openai_compat"),
      "| strong:", os.environ["MODEL_STRONG"])

## create_agent

**One current name, one deprecated one. The stack says so itself.**

- `create_react_agent` still imports silently; the complaint fires at call time
- the replacement is named right in the warning text
- tutorials built on the old name date themselves in one line

In [ ]:
import warnings

from langgraph.prebuilt import create_react_agent  # the import itself stays silent

model = chat_model("strong")

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    create_react_agent(model, [])  # the warning fires here, at call time

# pick by category: an unrelated warning may be recorded first
w = next(w for w in caught if "Deprecated" in w.category.__name__)
print(w.category.__name__)
print(w.message)

**A currency desk: one frozen rate table, two tools, zero network.**

- misses return advice, not exceptions: session 1's rule, third session running
- `currency_` prefix on both names; the reason lands later today
- `parse_docstring=True` on the second tool; what it buys gets printed later

In [ ]:
from langchain_core.tools import tool

# a fake rate table: no network, same numbers every run
RATES = {"EUR": 1.09, "GBP": 1.27, "JPY": 0.0064, "CHF": 1.13, "USD": 1.0}


@tool  # the docstring is the description the model reads
def currency_rate(code: str) -> str:
    """Return the USD rate for one ISO currency code, for example EUR."""
    code = code.strip().upper()
    if code not in RATES:
        # a miss the model can act on, not an exception
        return f"Unknown code {code!r}. Known codes: {', '.join(sorted(RATES))}."
    return f"1 {code} = {RATES[code]} USD"


@tool(parse_docstring=True)  # lifts the Args: lines onto the wire
def currency_convert(amount: float, source: str, target: str) -> str:
    """Convert an amount between two currencies via USD.

    Args:
        amount: The amount in the source currency.
        source: ISO code the amount is in, for example EUR.
        target: ISO code to convert into, for example JPY.
    """
    source, target = source.strip().upper(), target.strip().upper()
    unknown = [c for c in (source, target) if c not in RATES]
    if unknown:
        return f"Unknown codes: {', '.join(unknown)}. Known: {', '.join(sorted(RATES))}."
    return f"{amount} {source} = {amount * RATES[source] / RATES[target]:.2f} {target}"


TOOLS = [currency_rate, currency_convert]
print(currency_rate.invoke({"code": "eur"}))
print(currency_rate.invoke({"code": "XAU"}))  # the miss is a sentence, not a crash

**The whole session-2 agent, one call.**

- a model instance, a tools list, a system prompt
- `invoke` and `recursion_limit` unchanged: this compiles to the same graph

In [ ]:
from langchain.agents import create_agent

PROMPT = "You are a currency desk assistant. Use the tools for every rate."

agent = create_agent(model, TOOLS, system_prompt=PROMPT)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the euro worth in dollars?"}]},
    config={"recursion_limit": 8},  # session-2 discipline, unchanged
)

for message in result["messages"]:
    message.pretty_print()

## The graph underneath

**Four messages: session 2's transcript, out of code you did not write.**

- human, tool call, tool result, answer: the loop ran once and closed
- prove the rest of the claim: rebuild session 2 in twelve lines, print both graphs
- the rebuild is never invoked today; it exists to be drawn

In [ ]:
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode

bound = model.bind_tools(TOOLS)  # print-only rebuild: nothing here is invoked


def call_model(state: MessagesState) -> dict:
    return {"messages": [bound.invoke(state["messages"])]}


def should_continue(state: MessagesState) -> str:
    return "tools" if state["messages"][-1].tool_calls else END


builder = StateGraph(MessagesState)
builder.add_node("model", call_model)
builder.add_node("tools", ToolNode(TOOLS))
builder.add_edge(START, "model")
builder.add_conditional_edges("model", should_continue, ["tools", END])
builder.add_edge("tools", "model")

print("--- session 2, by hand ---")
print(builder.compile().get_graph().draw_mermaid())
print("--- create_agent ---")
print(agent.get_graph().draw_mermaid())

**Node for node, the same machine.**

- `model` and `tools`: the names match exactly, the edges tell the same loop
- one cosmetic difference: the `tools -> model` edge renders dashed in one drawing, solid in the other
- the docs teach the opposite order: start prebuilt, drop to graphs when stuck
- this course went bottom-up on purpose: session 2 is what this call compiles to

## content_blocks

**Every provider shapes `.content` differently. One typed view on top.**

- a text turn: `[{"type": "text", ...}]`; a tool turn: `[{"type": "tool_call", ...}]`
- `.content` stays raw and provider-shaped; nothing is rewritten
- read blocks by their `type` field and your code stops guessing

In [ ]:
tool_turn = result["messages"][1]  # the reply that asked for a tool
final_turn = result["messages"][-1]

print("tool turn  .content:       ", repr(tool_turn.content))
print("tool turn  .content_blocks:", tool_turn.content_blocks)
print("final turn .content:       ", repr(final_turn.content))
print("final turn .content_blocks:", final_turn.content_blocks)

## Middleware

**Four hooks around a loop you no longer own.**

- `before_model`, `wrap_model_call`, `wrap_tool_call`, `after_model`
- a decorator turns a plain function into agent middleware
- the `wrap_` pair receives a handler: call it, or refuse to
- this is where `create_agent` hands back the control the graph gave you

In [ ]:
from langchain.agents.middleware import before_model, wrap_model_call, wrap_tool_call


@before_model  # runs before every request the loop sends
def count_context(state, runtime):
    print(f"[before_model] sending {len(state['messages'])} messages")


@wrap_model_call  # sees exactly what leaves for the provider
def show_wire(request, handler):
    names = [t.name for t in request.tools]
    print(f"[wrap_model_call] {len(request.messages)} messages, tools {names}")
    return handler(request)


@wrap_tool_call  # wraps every tool execution
def audit_tool(request, handler):
    print(f"[wrap_tool_call] {request.tool_call['name']}({request.tool_call['args']})")
    return handler(request)  # run it for real; a veto would return here instead


watched = create_agent(chat_model("strong"), TOOLS, system_prompt=PROMPT,
                       middleware=[count_context, show_wire, audit_tool])

reply = watched.invoke(
    {"messages": [{"role": "user", "content": "How many dollars is one pound?"}]},
    config={"recursion_limit": 8},
)
print("final:", reply["messages"][-1].content)

**Prebuilt middleware: three names to know, a catalog behind them.**

- `PIIMiddleware`, next cell: scrub patterns before they reach the provider
- `SummarizationMiddleware`: session 4, when the context outgrows the window
- `HumanInTheLoopMiddleware`: session 6, approval before dangerous tools
- deeper in the catalog: retries, fallbacks, call limits, todo lists

In [ ]:
from langchain.agents.middleware import PIIMiddleware

guarded = create_agent(chat_model("cheap"), [], middleware=[PIIMiddleware("email")],
                       system_prompt="You are a currency desk assistant.")

out = guarded.invoke(
    {"messages": [{"role": "user", "content": "Mail the sheet to dana.lee@example.edu."}]},
    config={"recursion_limit": 8},
)
print(out["messages"][0].content)  # the user turn as the provider saw it

## The wire

**The schema was always there. Today it gets printed.**

- session 1 typed this dict by hand; session 2 let `bind_tools` hide it
- `@tool` reads only the signature and the docstring
- the dict rides in the request body of every call, the whole conversation long

In [ ]:
import json

from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_openai import ChatOpenAI

print(json.dumps(convert_to_openai_tool(currency_rate), indent=2))

# the real client class; constructing it sends nothing anywhere
client_model = ChatOpenAI(model=os.environ["MODEL_CHEAP"],
                          api_key=os.environ["LLM_API_KEY"],
                          base_url=os.getenv("LLM_BASE_URL"))
wire_bound = client_model.bind_tools(TOOLS)

# OpenAI envelope on purpose; a Gemini client wraps the same schema
print(list(wire_bound.kwargs))  # bind_tools only stored request kwargs
print([entry["function"]["name"] for entry in wire_bound.kwargs["tools"]])

In [ ]:
@tool  # same function, same docstring, flag removed
def convert_plain(amount: float, source: str, target: str) -> str:
    """Convert an amount between two currencies via USD.

    Args:
        amount: The amount in the source currency.
        source: ISO code the amount is in, for example EUR.
        target: ISO code to convert into, for example JPY.
    """
    return currency_convert.func(amount, source, target)


print("plain @tool:")
print(json.dumps(convert_to_openai_tool(convert_plain)["function"]["parameters"], indent=2))
print("with parse_docstring=True:")
print(json.dumps(convert_to_openai_tool(currency_convert)["function"]["parameters"], indent=2))

**Name and description are prompt. One flag decides how much of yours ships.**

- plain `@tool`: the `Args:` lines sit uselessly in the description blob
- `parse_docstring=True`: each parameter carries its own `description`
- you wrote that text for the model; check it actually leaves the building

## Tool design

**Anthropic's rules for tools agents can actually use.**

- consolidate: one capable tool beats three fragments the model must chain
- one prefix per domain: `currency_*` keeps kin together and collisions out
- semantic identifiers: `EUR` reads back; a UUID is copy-noise the model fumbles
- the next rule can be measured instead of trusted:

In [ ]:
from langchain_core.messages import ToolMessage
from langchain_core.messages.utils import count_tokens_approximately

verbose = ToolMessage(
    content=json.dumps({
        "status": "ok",
        "query": {"code": "EUR"},
        "result": {"base": "EUR", "quote": "USD", "rate": 1.09,
                   "retrieved_at": "2026-08-11T09:00:00Z", "source": "desk-cache"},
    }),
    tool_call_id="demo",  # both messages state the same single fact
)
concise = ToolMessage(content="1 EUR = 1.09 USD", tool_call_id="demo")

print("verbose reply:", count_tokens_approximately([verbose]), "tokens")
print("concise reply:", count_tokens_approximately([concise]), "tokens")

**Same fact, several times the tokens. Every later turn pays again.**

- a tool reply enters the history and is resent with every following request
- economical responses: filter, paginate, truncate before you return
- actionable errors complete the list: `find_book` and `care_notes` lived this rule already

## Known failure modes

**Small models fail in known ways. Design for both.**

- hallucinated tool names: `currencyRate` where `currency_rate` exists
- arguments that violate the schema: a word where it says number
- the committed run replays the miss from a script; your live model may sail through

In [ ]:
clumsy = create_agent(chat_model("cheap"), TOOLS, system_prompt=PROMPT)

recovery = clumsy.invoke(
    {"messages": [{"role": "user", "content": "What is the yen worth in dollars?"}]},
    config={"recursion_limit": 10},  # a failure mode costs extra turns; cap them
)

for message in recovery["messages"]:
    message.pretty_print()

**When the name misses, it comes back as a `ToolMessage`, not a crash.**

- "not a valid tool, try one of [...]": the model reads it and self-corrects
- that registry check ships inside `ToolNode`, session 2's and today's alike
- writing your own executor? the check is yours to add: `wrap_tool_call` is the hook

## Cost

**Counters today, prices never. Your provider's price page is homework.**

- every reply carries `usage_metadata`: input, output, total tokens
- cached input has its own counter: a repeated prefix is billed differently
- the trace shows the same counters per node, summed per run

In [ ]:
final = result["messages"][-1]  # the closing reply of the first run

print("usage_metadata:", final.usage_metadata)
print("cached input:  ", final.usage_metadata.get("input_token_details"))
print("response_metadata keys:", list(final.response_metadata))

**Silent spend, now visible as numbers.**

- session 1's crash: the budget burned on hidden reasoning, `content` empty
- in counters: output tokens spent, answer text blank
- an agent that "did nothing" still paid; the counters are where you catch it

## The data rule

**The rule is about tool outputs, not what you type.**

- whatever a tool returns enters the context and goes to the provider
- files, pages, letters, other APIs' replies: all of it travels
- fictional data only, all semester; `PIIMiddleware` is a seatbelt, not permission

## The trace

**Same handler as session 2. New things to read in it.**

- one span per tool call: name, arguments, reply
- per-node token counters and how long each step took, and the run totals they sum to
- `flush()` or nothing sends: a notebook kernel never exits on its own

In [ ]:
from langfuse import get_client
from langfuse.langchain import CallbackHandler

client = get_client()  # reads LANGFUSE_HOST and both keys from the environment
print("server:", os.getenv("LANGFUSE_HOST"), "| up:", client.auth_check())

handler = CallbackHandler()  # a fresh handler per run gives one trace per run
traced = agent.invoke(
    {"messages": [{"role": "user", "content": "Turn 250 francs into yen."}]},
    config={"recursion_limit": 8, "callbacks": [handler]},
)

client.flush()  # the classic notebook mistake is forgetting this line

print(traced["messages"][-1].content)

**Open the trace and compute what this run cost.**

- find the priciest model call; the counters say why it is that one
- multiply by your provider's price page, not by a number from a slide
- the input counter grows every turn: that is the history being resent

## Practice

**Rebuild your assistant on `create_agent`, in your own repository.**

1. the session-2 graph stays in git history; the agent becomes one call
2. three full tools of your own domain: one name prefix, actionable misses
3. print the schema dict of each tool and read what the model will read
4. required middleware: a `wrap_tool_call` that logs every call's name and args

**Then run one traced conversation and read it with the counters open.**

5. ask something that takes at least two tool calls to answer
6. find the priciest model call in the trace and say why
7. cost arithmetic in one sentence, from your provider's price page
8. empty Langfuse? call `flush()` first, then start debugging

**Required artifact: `runs/session-03.md`, committed.**

- the printed schema dicts of your three tools
- the transcript of the traced conversation
- your one-sentence cost arithmetic
- the exported trace, committed next to it

**Stretch, if you finish early.**

- a `wrap_tool_call` that vetoes one tool by name and tells the model why
- split one tool into three, ask the same question, compare the traces
- the split run's extra tokens are the consolidate rule, measured on your own agent

**The project clock starts today.**

- the topic set opens; topic approval and review pairs follow in the next two sessions
- `create_agent` plus middleware is the recommended starting architecture
- tool quality enters the defense criteria; today's rules are the checklist
- tracing is mandatory from now on, for everything that calls a model

## Next time

**Context management: long dialogs against a finite window — trimming, summarization, and state that survives a restart.**